# 1. Import & Constructor — Lasso Regression

---

# Required Imports

```python
from sklearn.linear_model import Lasso
```

A complete workflow usually includes:

```python
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import Lasso
```

> **Note:** Like Ridge Regression, **Lasso is highly sensitive to feature scaling**. Always standardize numerical features before training.

---

# Constructor Syntax

```python
Lasso(
    alpha=1.0,
    fit_intercept=True,
    precompute=False,
    copy_X=True,
    max_iter=1000,
    tol=1e-4,
    warm_start=False,
    positive=False,
    random_state=None,
    selection='cyclic'
)
```

---

# Important Parameters

| Parameter       | Default    | Description                                                 | Common Usage                                         |
| --------------- | ---------- | ----------------------------------------------------------- | ---------------------------------------------------- |
| `alpha`         | `1.0`      | Strength of L1 regularization                               | `0.001`, `0.01`, `0.1`, `1`, `10`                    |
| `fit_intercept` | `True`     | Whether to calculate the intercept                          | Usually `True`                                       |
| `precompute`    | `False`    | Whether to precompute the Gram matrix to speed up fitting   | Usually leave as default                             |
| `copy_X`        | `True`     | Whether to copy the input data                              | Usually leave as default                             |
| `max_iter`      | `1000`     | Maximum number of optimization iterations                   | Increase if convergence warnings occur               |
| `tol`           | `1e-4`     | Tolerance for stopping optimization                         | Usually keep default                                 |
| `warm_start`    | `False`    | Reuse coefficients from the previous fit                    | Useful when fitting repeatedly with similar settings |
| `positive`      | `False`    | Restrict coefficients to be non-negative                    | Use only when required                               |
| `random_state`  | `None`     | Controls randomness when `selection='random'`               | Set for reproducibility                              |
| `selection`     | `'cyclic'` | Order in which coefficients are updated during optimization | `'cyclic'` or `'random'`                             |

---

# Parameter Explanation

---

## 1. `alpha`

```python
Lasso(alpha=1.0)
```

Controls the strength of the L1 penalty.

Examples:

```python
Lasso(alpha=0.001)
Lasso(alpha=0.01)
Lasso(alpha=0.1)
Lasso(alpha=1)
Lasso(alpha=10)
```

General behavior:

* Smaller `alpha` → Less regularization
* Larger `alpha` → More coefficients become zero
* Too large → Underfitting

---

## 2. `fit_intercept`

```python
Lasso(fit_intercept=True)
```

Determines whether the intercept (bias) is learned.

Model equation:

[
\hat y=w_1x_1+w_2x_2+\cdots+w_nx_n+b
]

If

```python
fit_intercept=False
```

then

[
\hat y=w_1x_1+w_2x_2+\cdots+w_nx_n
]

Use `False` only when the data has already been centered.

---

## 3. `precompute`

```python
Lasso(precompute=False)
```

Controls whether the **Gram matrix** is precomputed.

The Gram matrix is:

[
G = X^T X
]

It is used internally during optimization.

* `False` → Compute it only when needed (default).
* `True` → Compute it once in advance, which can speed up fitting for some dense datasets.

For most users:

```python
precompute=False
```

is recommended.

---

## 4. `copy_X`

```python
Lasso(copy_X=True)
```

* `True` → Creates a copy of the input data.
* `False` → May modify the original array internally to save memory.

Usually leave it as the default.

---

## 5. `max_iter`

```python
Lasso(max_iter=1000)
```

Maximum number of optimization iterations.

Unlike Ridge, **Lasso does not have a `solver` parameter**. Scikit-learn uses **coordinate descent** internally to optimize the Lasso objective.

If optimization does not converge, you may see a warning such as:

```text
ConvergenceWarning:
Objective did not converge.
```

In that case, increase the iteration limit:

```python
Lasso(max_iter=5000)
```

---

## 6. `tol`

```python
Lasso(tol=1e-4)
```

Tolerance used to determine when optimization should stop.

Smaller value:

```python
tol=1e-6
```

* More accurate convergence
* More iterations
* Slower training

Larger value:

```python
tol=1e-2
```

* Faster training
* Less precise convergence

---

## 7. `warm_start`

```python
Lasso(warm_start=True)
```

Normally, every call to `fit()` starts with coefficients initialized to zero.

With:

```python
warm_start=True
```

the model starts from the coefficients learned during the previous `fit()` call.

**warm_start** reuses the coefficients from the previous fit() call on the same model object. It does not reuse coefficients from a different model object.

Example:

```python
model = Lasso(alpha=1.0, warm_start=True)

model.fit(X_train, y_train)

model.set_params(alpha=0.8)

model.fit(X_train, y_train)
```

This can reduce training time when fitting several similar models.

---

## 8. `positive`

```python
Lasso(positive=True)
```

Restricts all coefficients to be non-negative.

Without constraint:

```text
Age       = -8.5
Income    = 15.3
Salary    = 7.9
```

With `positive=True`:

```text
Age       = 0.0 or positive
Income    = positive
Salary    = positive
```

Use this only when negative coefficients are not meaningful for your application.

---

## 9. `random_state`

```python
Lasso(
    selection="random",
    random_state=42
)
```

Used only when:

```python
selection="random"
```

Setting a fixed value makes the optimization reproducible.

If `selection="cyclic"` (the default), `random_state` has no effect.

---

## 10. `selection`

```python
Lasso(selection="cyclic")
```

Controls the order in which coefficients are updated during coordinate descent.

Two options are available:

### `"cyclic"` (default)

Updates coefficients in a fixed order.

```text
Feature1
   ↓
Feature2
   ↓
Feature3
   ↓
Feature4
   ↓
Repeat
```

Advantages:

* Deterministic
* Stable
* Recommended for most use cases

---

### `"random"`

Randomly selects a coefficient to update at each step.

```text
Feature3
   ↓
Feature1
   ↓
Feature4
   ↓
Feature2
```

Advantages:

* Can converge faster on some large datasets.

Requires:

```python
random_state=42
```

for reproducible results.

---

# Recommended Settings

For most regression problems:

```python
lasso = Lasso(
    alpha=1.0
)
```

For larger datasets or when tuning:

```python
lasso = Lasso(
    alpha=0.1,
    max_iter=5000
)
```

Instead of manually guessing `alpha`, use **`GridSearchCV`** or **`LassoCV`** to find the optimal value.

# 2. Methods & Attributes — Lasso Regression

One advantage of learning Ridge first is that **Lasso follows exactly the same scikit-learn estimator API**. Almost all methods and attributes are identical.

---

# Methods Table

| Method         | Purpose                                        | Returns         |
| -------------- | ---------------------------------------------- | --------------- |
| `fit(X, y)`    | Trains the Lasso Regression model              | `self`          |
| `predict(X)`   | Predicts target values for new data            | `numpy.ndarray` |
| `score(X, y)`  | Computes the R² (Coefficient of Determination) | `float`         |
| `get_params()` | Returns the model's parameters                 | `dict`          |
| `set_params()` | Updates one or more parameters                 | `self`          |

---

# Attributes Table

| Attribute           | Description                                                                            |
| ------------------- | -------------------------------------------------------------------------------------- |
| `coef_`             | Learned coefficients (weights) for each feature                                        |
| `intercept_`        | Learned intercept (bias term)                                                          |
| `n_iter_`           | Number of iterations taken for optimization to converge                                |
| `n_features_in_`    | Number of input features seen during training                                          |
| `feature_names_in_` | Names of input features (if trained using a pandas DataFrame with string column names) |

> **Difference from Ridge:** Lasso exposes an additional commonly used attribute, **`n_iter_`**, because it is trained using an **iterative Coordinate Descent optimizer**.
---